In [ ]:
#Data Preparation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from docx import Document
from docx.shared import Pt, Inches
from pathlib import Path


def _find_root(markers=('us_data', 'mexico_data')):
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if all((cand / m).exists() for m in markers):
            return cand
    raise FileNotFoundError(
        'Could not locate the repo root (expected folders us_data/ and '
        f'mexico_data/). Searched upward from {here}.')

ROOT = _find_root()
RESULTS = ROOT / 'us' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)


SEX_TO_FEMALE = {'Women+': 1, 'Women': 1, 'Female': 1,
                 'Men+': 0,  'Men': 0,   'Male': 0}

def recode_sex(s):
    """Map a raw Sex label series onto the {0 = male, 1 = female} indicator."""
    out = s.astype(str).str.strip().map(SEX_TO_FEMALE)
    if out.isna().any():
        bad = sorted(s[out.isna()].astype(str).str.strip().unique())
        raise ValueError(f"Unrecognised Sex label(s): {bad}")
    return out.astype(int)


df = pd.read_csv(ROOT / 'us_data' / 'aggregatedData.csv').rename(columns={
    'State': 'state', 'Year': 'year',
    'BusinessOwnersPSTS': 'business_owners_psts', 'LaborForce': 'labor_force'})
df['female'] = recode_sex(df['Sex'])
df['gender'] = df['female'].map({0: 'Male', 1: 'Female'})

df['biz_rate'] = df['business_owners_psts'] / df['labor_force']
df['log_rate'] = np.log(df['biz_rate'])

df_final = df[df['year'].between(2014, 2023)].dropna(subset=['log_rate', 'labor_force']).copy()
df_final['year'] = df_final['year'].astype(int)

In [ ]:
#Parallel Trends

sns.set_theme(style="whitegrid")

df_pre = df_final[df_final['year'] <= 2017].copy()

plt.figure(figsize=(10, 6))
sns.lineplot(data=df_final, x='year', y='log_rate', hue='gender', marker='o', linewidth=2.5)

plt.axvline(x=2018, color='red', linestyle='--', label='Treatment cutoff (2018)')
plt.title('Parallel Trends Check: Development of Business Ownership (log Rate)', fontsize=14)
plt.ylabel('Log(Business Owners / Labor Force)')
plt.legend()
plt.savefig(RESULTS / 'parallelTrends.png', dpi=300, bbox_inches='tight')
plt.show()

model_pre = smf.ols("log_rate ~ female * C(year) + C(state)", data=df_pre)
res_pre = model_pre.fit(cov_type='cluster', cov_kwds={'groups': df_pre['state']})

print(res_pre.summary())

interaction_vars = [var for var in res_pre.params.index if 'female:C(year)' in var]
f_test = res_pre.f_test(interaction_vars)

print(f_test.summary())

doc = Document()
section = doc.sections[0]
section.page_width = Inches(11)
section.page_height = Inches(8.5)
doc.add_heading('OLS Parallel Trends Regression Results', 0)
paragraph = doc.add_paragraph()
run = paragraph.add_run(str(res_pre.summary()))
run.font.name = 'Courier New'
run.font.size = Pt(9)
doc.save(str(RESULTS / "Parallel_Trends_OLS.docx"))

doc = Document()
doc.add_heading('F Test Results', 0)
paragraph = doc.add_paragraph()
run = paragraph.add_run(str(f_test.summary()))
run.font.name = 'Courier New'
run.font.size = Pt(9)
doc.save(str(RESULTS / "F_Test_Results.docx"))